# 03 — Monte-Carlo Miss Distance & CEP

Run (or load) a Monte-Carlo batch, compute the **CEP** (Circular Error Probable — the median miss distance), and tie the dispersion back to the characterized sensor noise from the Allan-variance pipeline (`configs/sensor_params.json`).

In [1]:
import sys
from pathlib import Path

# Repo-relative imports (works whether run from repo root or notebooks dir).
REPO = Path.cwd()
while not (REPO / "build-native").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "postproc"))
sys.path.insert(0, str(REPO / "sensors"))

import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline


In [2]:
import json
from gncpost.loaders import run_cli, load_summary
from gncpost.montecarlo import compute_stats, plot_cep

batch = REPO / 'runs' / 'mc_batch'
if not (batch / 'summary.csv').exists():
    run_cli(REPO / 'configs' / 'montecarlo.json', batch, timeout=180)
summary = load_summary(batch)
stats = compute_stats(summary)
print(stats.summary_line())

n=200  CEP=0.56 m  mean=0.63 m  std=0.45 m  P90=1.21 m  RMS=0.77 m  intercept=100.0%


## Miss-distance distribution + CEP
The dashed line is the CEP (50% of impacts fall within it); the dotted line is the mean.

In [3]:
fig = plot_cep(summary, stats); plt.show()

/tmp/ipykernel_41029/568054481.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plot_cep(summary, stats); plt.show()


## Tie to the characterized sensor noise
The Monte-Carlo runs use the IMU + seeker noise in `configs/sensor_params.json`, which was produced by the Allan-variance characterization in `sensors/`. The CEP below is the terminal-accuracy consequence of that noise budget.

In [4]:
imu = json.loads((REPO / 'configs' / 'sensor_params.json').read_text())['imu']
print('IMU noise driving these results (from Allan characterization):')
print(f"  accel white (VRW): {imu['accel_white']:.3e} (m/s^2)/sqrt(Hz)")
print(f"  accel bias instab (sigma_GM): {imu['accel_bias_instability']:.3e} m/s^2")
print(f"  gyro  white (ARW): {imu['gyro_white']:.3e} (rad/s)/sqrt(Hz)")
print()
print(f'  -> CEP = {stats.cep:.2f} m, P90 = {stats.p90:.2f} m over {stats.n} cases')

IMU noise driving these results (from Allan characterization):
  accel white (VRW): 5.997e-04 (m/s^2)/sqrt(Hz)
  accel bias instab (sigma_GM): 4.342e-04 m/s^2
  gyro  white (ARW): 8.998e-05 (rad/s)/sqrt(Hz)

  -> CEP = 0.56 m, P90 = 1.21 m over 200 cases
